In [3]:
!pip install -q langchain langchain-community langchain-openai langgraph groq

from groq import Groq
import os
import asyncio
from typing import Optional

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    Runnable,
    RunnableParallel,
    RunnablePassthrough,
)

load_dotenv()

True

In [4]:
try:
    llm: Optional[ChatOpenAI] = ChatOpenAI(
        model="llama3-70b-8192",   # ✅ Groq model
        temperature=0.7,
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
    )
except Exception as e:
    print(f"Error initializing LLM: {e}")
    llm = None


# --- Define Independent Chains ---

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# --- Parallel Execution Block ---

map_chain = RunnableParallel({
    "summary": summarize_chain,
    "questions": questions_chain,
    "key_terms": terms_chain,
    "topic": RunnablePassthrough()
})


# --- Synthesis Prompt ---

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:

Summary: {summary}
Related Questions: {questions}
Key Terms: {key_terms}

Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}")
])


# --- Full Chain ---

full_parallel_chain = (
    map_chain
    | synthesis_prompt
    | llm
    | StrOutputParser()
)


# --- Run the Chain ---

async def run_parallel_example(topic: str) -> None:
    if not llm:
        print("LLM not initialized. Cannot run example.")
        return

    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")

    try:
        response = await full_parallel_chain.ainvoke(topic)

        print("\n--- Final Response ---")
        print(response)

    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")


if __name__ == "__main__":
    test_topic = "The history of space exploration"
    asyncio.run(run_parallel_example(test_topic))

RuntimeError: asyncio.run() cannot be called from a running event loop